In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType

# =========================
# CONFIG
# =========================
STORAGE_ACCOUNT = "ragadziyada"
STORAGE_KEY = "qIXjwo7EGK8an84BPCk446tqY9L7K7r3a9W2WB+voe2vUCvW1SK3qc/7UGOicKyBAtrptYcVfTB7+AStvWZi0A=="

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

ARTICLES_IN = f"abfss://processed-p1@{STORAGE_ACCOUNT}.dfs.core.windows.net/articles_landing/"
CUSTOMERS_IN = f"abfss://processed-p1@{STORAGE_ACCOUNT}.dfs.core.windows.net/customers_landing/"
TRANSACTIONS_IN = f"abfss://processed-p1@{STORAGE_ACCOUNT}.dfs.core.windows.net/transactions_landing/"

ARTICLES_OUT = f"abfss://processed-p1@{STORAGE_ACCOUNT}.dfs.core.windows.net/articles_clean/"
CUSTOMERS_OUT = f"abfss://processed-p1@{STORAGE_ACCOUNT}.dfs.core.windows.net/customers_clean/"
TRANSACTIONS_OUT = f"abfss://processed-p1@{STORAGE_ACCOUNT}.dfs.core.windows.net/transactions_clean/"

# =========================
# LOAD
# =========================
articles_df = spark.read.parquet(ARTICLES_IN)
customers_df = spark.read.parquet(CUSTOMERS_IN)
transactions_df = spark.read.parquet(TRANSACTIONS_IN)

# =========================
# CLEAN ARTICLES
# =========================
articles_clean = articles_df.dropDuplicates(["article_id"])

for c in articles_clean.columns:
    if c == "article_id":
        articles_clean = articles_clean.withColumn(c, F.col(c).cast(StringType()))
    else:
        articles_clean = articles_clean.withColumn(c, F.col(c).cast(StringType()))

fill_unknown_cols = [
    "prod_name", "product_type_name", "product_group_name", "graphical_appearance_name",
    "colour_group_name", "perceived_colour_value_name", "perceived_colour_master_name",
    "department_name", "index_name", "index_group_name", "section_name",
    "garment_group_name", "detail_desc"
]

for c in fill_unknown_cols:
    if c in articles_clean.columns:
        articles_clean = articles_clean.fillna({c: "Unknown"})

# =========================
# CLEAN CUSTOMERS
# =========================
customers_clean = customers_df.dropDuplicates(["customer_id"])

customers_clean = customers_clean.withColumn("customer_id", F.col("customer_id").cast(StringType()))

if "age" in customers_clean.columns:
    customers_clean = customers_clean.withColumn("age", F.col("age").cast(IntegerType()))
    customers_clean = customers_clean.withColumn(
        "age",
        F.when((F.col("age") < 12) | (F.col("age") > 100), None).otherwise(F.col("age"))
    )

for c in ["FN", "Active"]:
    if c in customers_clean.columns:
        customers_clean = customers_clean.withColumn(
            c,
            F.coalesce(
                F.col(c).cast("double").cast("int"),
                F.lit(0)
            )
        )

if "fashion_news_frequency" in customers_clean.columns:
    customers_clean = customers_clean.withColumn(
        "fashion_news_frequency",
        F.lower(F.trim(F.col("fashion_news_frequency").cast(StringType())))
    )
    customers_clean = customers_clean.withColumn(
        "fashion_news_frequency",
        F.when(F.col("fashion_news_frequency").isin("regularly", "regularly."), "regularly")
         .when(F.col("fashion_news_frequency") == "monthly", "monthly")
         .when(F.col("fashion_news_frequency").isin("none", "none."), "none")
         .otherwise(F.coalesce(F.col("fashion_news_frequency"), F.lit("unknown")))
    )

# =========================
# CLEAN TRANSACTIONS
# =========================
transactions_clean = transactions_df \
    .withColumn("customer_id", F.col("customer_id").cast(StringType())) \
    .withColumn("article_id", F.col("article_id").cast(StringType())) \
    .withColumn("price", F.col("price").cast(DoubleType())) \
    .withColumn("sales_channel_id", F.col("sales_channel_id").cast(IntegerType())) \
    .withColumn("t_dat", F.to_date(F.col("t_dat"))) \
    .dropDuplicates(["t_dat", "customer_id", "article_id", "price", "sales_channel_id"])

# remove invalid / incomplete transaction rows
transactions_before = transactions_df.count()

transactions_clean = transactions_clean.filter(
    F.col("customer_id").isNotNull() &
    F.col("article_id").isNotNull() &
    F.col("t_dat").isNotNull() &
    F.col("price").isNotNull() &
    (F.col("price") > 0)
)

transactions_after = transactions_clean.count()

print("transactions before cleaning:", transactions_before)
print("transactions after cleaning:", transactions_after)
print("transactions removed:", transactions_before - transactions_after)

transactions_clean = transactions_clean \
    .withColumn("year", F.year("t_dat")) \
    .withColumn("month", F.month("t_dat")) \
    .withColumn("week", F.weekofyear("t_dat")) \
    .withColumn("day_of_week", F.dayofweek("t_dat"))

# =========================
# WRITE CLEAN DATA
# =========================
articles_clean.write.mode("overwrite").parquet(ARTICLES_OUT)
customers_clean.write.mode("overwrite").parquet(CUSTOMERS_OUT)
transactions_clean.write.mode("overwrite").parquet(TRANSACTIONS_OUT)

print("Clean datasets written successfully.")
print("articles_clean:", articles_clean.count())
print("customers_clean:", customers_clean.count())
print("transactions_clean:", transactions_clean.count())

transactions before cleaning: 31788324
transactions after cleaning: 28813419
transactions removed: 2974905
Clean datasets written successfully.
articles_clean: 105542
customers_clean: 1371980
transactions_clean: 28813419
